# Training Mask2Former (Swin-Tiny)

**Tugas 3 - Segmentasi Instance** | keluarga arsitektur: **transformer (Hugging Face transformers)**

Notebook ini melatih model pada **Panoptic Indoor Segmentation v5** dengan variabel NRP
**jumlah data latih 25% / 50% / 100%** (hanya split `train` yang dikurangi; `valid` dan `test` v5 tetap utuh),
lalu mengevaluasi setiap run di **valid**, **test asli v5**, dan **test set foto sendiri** (`mycapture`).

Semua model memakai evaluasi yang sama: **pycocotools** (mask & box AP), `MAX_DET` dan `SCORE_THRESH` yang sama.

### Setup Kaggle (sekali saja)
1. **Settings → Accelerator → GPU T4 x2** (notebook ini memakai 1 GPU). Hindari P100: PyTorch versi baru tidak lagi mendukung GPU P100.
2. **Settings → Internet → On** (butuh verifikasi nomor HP di akun Kaggle).
3. **Add-ons → Secrets → Add secret** dengan nama `ROBOFLOW_API_KEY` berisi Private API Key Roboflow, lalu centang untuk notebook ini.
4. Jalankan dengan **Save Version → Save & Run All (Commit)** supaya training jalan di background (browser boleh ditutup).
   Hasil ada di tab **Output**: folder `results_mask2former` dan file `.zip`-nya.

Mask2Former tidak memakai proposal/box: 100 *query* transformer masing-masing menghasilkan satu kelas dan satu mask (resolusi 1/4 input, lalu di-upsample). Bandingkan pola kegagalannya dengan YOLO dan Mask R-CNN.

Model ini paling berat. Kalau muncul `loss tidak valid` terus-menerus, ubah `USE_AMP = False`.

In [ ]:
%pip install -q roboflow pycocotools

In [ ]:
%%writefile common.py
"""Kode bersama untuk notebook training Tugas 3 (Panoptic Indoor v5).

Isi: download data, penyeragaman kelas, subset train bertingkat (25/50/100%),
evaluasi COCO (pycocotools) yang sama untuk semua model, log waktu GPU, dan overlay prediksi.
File ini ditulis ulang oleh cell %%writefile di setiap notebook, jadi isinya identik di ketiga notebook.
"""
import os, json, time, random, shutil, csv, datetime, math
from collections import defaultdict, Counter

import numpy as np
from PIL import Image

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_WINDOWS = os.name == "nt"


# ----------------------------------------------------------------------------- util umum
def get_secret(name):
    """Ambil secret dari Kaggle Secrets / Colab Secrets / env var / input manual."""
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v.strip()
    except Exception:
        pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    if ON_KAGGLE:
        raise RuntimeError(f"Secret '{name}' tidak ditemukan. Tambahkan lewat Add-ons -> Secrets, "
                           "lalu centang/attach ke notebook ini.")
    from getpass import getpass
    return getpass(f"{name}: ").strip()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, p):
    os.makedirs(os.path.dirname(os.path.abspath(p)), exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f)


def append_csv(path, row):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    rows, fields = [], []
    if os.path.exists(path):
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fields, rows = list(reader.fieldnames or []), list(reader)
    if all(k in fields for k in row):
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fields or list(row.keys()))
            if not fields:
                w.writeheader()
            w.writerow(row)
        return
    # ada kolom baru (mis. val_mask_AP baru muncul di epoch evaluasi): tulis ulang dengan header gabungan
    fields += [k for k in row if k not in fields]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows + [row])


def frac_tag(frac):
    return f"{int(round(frac * 100)):03d}pct"


def gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return "cpu"


class GpuTimer:
    """Mencatat waktu GPU per run ke results/gpu_log.csv (bukti proses untuk laporan)."""

    def __init__(self, results_dir, model, tag, note=""):
        self.path = os.path.join(results_dir, "gpu_log.csv")
        self.model, self.tag, self.note = model, tag, note

    def __enter__(self):
        self.t0 = time.time()
        self.start = datetime.datetime.now().isoformat(timespec="seconds")
        return self

    def __exit__(self, *exc):
        self.minutes = (time.time() - self.t0) / 60
        append_csv(self.path, {
            "model": self.model, "run": self.tag, "gpu": gpu_name(),
            "start": self.start, "end": datetime.datetime.now().isoformat(timespec="seconds"),
            "minutes": round(self.minutes, 2), "status": "error" if exc[0] else "ok", "note": self.note,
        })
        return False


# ----------------------------------------------------------------------------- download
def has_coco_split(d):
    return d and os.path.isdir(d) and any(
        os.path.exists(os.path.join(d, s, "_annotations.coco.json")) for s in ("train", "valid", "test"))


def download_roboflow(api_key, workspace, project, version, dest, fmt="coco-segmentation"):
    if has_coco_split(dest):
        print("Sudah ada:", dest)
        return dest
    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    ds = rf.workspace(workspace).project(project).version(version).download(fmt, location=dest, overwrite=False)
    return ds.location


# ----------------------------------------------------------------------------- kelas & konversi COCO
def real_categories(coco):
    """Buang kategori induk buatan Roboflow (mis. 'indoor-objects'), sisanya kategori asli."""
    cats = coco["categories"]
    supers = {c.get("supercategory") for c in cats}
    return [c for c in cats if not (c["name"] in supers and c.get("supercategory") in ("none", None, ""))]


def canonical_classes(v5_train_json, merge=None):
    """Daftar kelas final (urutan = urutan kategori v5). merge: {nama_lama: nama_baru atau None(=buang)}."""
    merge = merge or {}
    names = []
    for c in real_categories(load_json(v5_train_json)):
        n = merge.get(c["name"], c["name"])
        if n is not None and n not in names:
            names.append(n)
    return names


def _poly_ok(seg):
    return isinstance(seg, list) and any(len(p) >= 6 for p in seg)


def _resize_rle(seg, h, w, nh, nw):
    from pycocotools import mask as M
    rle = M.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list) else seg
    m = M.decode(rle)
    m = np.array(Image.fromarray(m * 255).resize((nw, nh), Image.NEAREST)) > 127
    out = M.encode(np.asfortranarray(m.astype(np.uint8)))
    out["counts"] = out["counts"].decode("ascii")
    return out


def to_canonical_coco(src_dirs, classes, out_json, merge=None, resized_dir=None, max_side=1024, target_side=640):
    """Gabung satu/lebih folder split Roboflow (COCO) jadi satu COCO dengan id kelas kanonik 1..K.

    - Nama kelas dipetakan lewat `merge`, kelas yang tidak ada di `classes` dibuang (dilaporkan).
    - Gambar yang sisi panjangnya > max_side diperkecil ke target_side (anotasi ikut diskalakan),
      supaya evaluasi foto HP resolusi penuh tidak kehabisan memori.
    - Setiap image punya key tambahan 'path' (path absolut file gambar).
    """
    merge = merge or {}
    name_to_id = {n: i + 1 for i, n in enumerate(classes)}
    images, anns, dropped = [], [], Counter()
    next_img, next_ann = 1, 1
    for d in src_dirs:
        src = load_json(os.path.join(d, "_annotations.coco.json"))
        cat_name = {c["id"]: c["name"] for c in src["categories"]}
        by_img = defaultdict(list)
        for a in src["annotations"]:
            by_img[a["image_id"]].append(a)
        for im in src["images"]:
            path = os.path.join(d, im["file_name"])
            if not os.path.exists(path):
                dropped["<file gambar hilang>"] += 1
                continue
            with Image.open(path) as pil:
                w, h = pil.size
            scale = 1.0
            if max(w, h) > max_side:
                scale = target_side / max(w, h)
                nw, nh = round(w * scale), round(h * scale)
                os.makedirs(resized_dir, exist_ok=True)
                new_path = os.path.join(resized_dir, os.path.basename(d.rstrip("/\\")) + "_" + im["file_name"])
                if not os.path.exists(new_path):
                    with Image.open(path) as pil:
                        pil.convert("RGB").resize((nw, nh), Image.BILINEAR).save(new_path, quality=95)
                path = new_path
            else:
                nw, nh = w, h
            img_id = next_img
            next_img += 1
            images.append({"id": img_id, "file_name": os.path.basename(path), "width": nw, "height": nh,
                           "path": os.path.abspath(path), "orig_file_name": im["file_name"]})
            for a in by_img[im["id"]]:
                name = cat_name.get(a["category_id"], "?")
                name = merge.get(name, name)
                # "<kelas>-crowd" (mis. chair-crowd): satu polygon untuk kerumunan objek yang tidak bisa dipisah.
                # Disimpan sebagai <kelas> dengan iscrowd=1, sehingga pycocotools memperlakukannya sebagai
                # area abaikan (prediksi di dalamnya tidak dihitung salah, tidak terdeteksi juga tidak dihitung luput).
                crowd = isinstance(name, str) and name.endswith("-crowd")
                if crowd:
                    name = name[: -len("-crowd")]
                if name is None or name not in name_to_id:
                    if name is not None:
                        dropped[name] += 1
                    continue
                seg = a.get("segmentation")
                if isinstance(seg, dict):
                    if scale != 1.0:
                        seg = _resize_rle(seg, h, w, nh, nw)
                elif _poly_ok(seg):
                    seg = [[float(v) * scale for v in p] for p in seg if len(p) >= 6]
                else:
                    dropped["<tanpa polygon>"] += 1
                    continue
                x, y, bw, bh = a["bbox"]
                anns.append({"id": next_ann, "image_id": img_id, "category_id": name_to_id[name],
                             "segmentation": seg, "bbox": [x * scale, y * scale, bw * scale, bh * scale],
                             "area": float(a.get("area", bw * bh)) * scale * scale, "iscrowd": int(crowd)})
                next_ann += 1
    coco = {"images": images, "annotations": anns,
            "categories": [{"id": i + 1, "name": n, "supercategory": "indoor"} for i, n in enumerate(classes)]}
    save_json(coco, out_json)
    if dropped:
        print(f"  [{os.path.basename(out_json)}] anotasi dibuang:", dict(dropped))
    return coco


# ----------------------------------------------------------------------------- subset bertingkat
def make_nested_subsets(train_coco, fractions, seed, out_dir):
    """Subset 25% ⊂ 50% ⊂ 100% dengan seed tetap.

    Urutan gambar: pertama, untuk setiap kelas (dari yang paling jarang) diambil satu gambar yang
    memuat kelas itu (supaya sebanyak mungkin kelas masih terwakili di subset terkecil), lalu sisa
    gambar diacak. Subset = prefix dari urutan ini, sehingga otomatis bertingkat.
    """
    rng = random.Random(seed)
    img_ids = sorted(im["id"] for im in train_coco["images"])
    img_cls, cls_imgs = defaultdict(set), defaultdict(set)
    for a in train_coco["annotations"]:
        img_cls[a["image_id"]].add(a["category_id"])
        cls_imgs[a["category_id"]].add(a["image_id"])
    order, chosen, covered = [], set(), set()
    for cid in sorted(cls_imgs, key=lambda c: (len(cls_imgs[c]), c)):
        if cid in covered:
            continue
        pick = rng.choice(sorted(cls_imgs[cid] - chosen))
        order.append(pick)
        chosen.add(pick)
        covered |= img_cls[pick]
    n_cover = len(order)
    rest = [i for i in img_ids if i not in chosen]
    rng.shuffle(rest)
    order += rest

    os.makedirs(out_dir, exist_ok=True)
    save_json({"seed": seed, "order": order, "n_coverage_images": n_cover}, os.path.join(out_dir, "train_order.json"))
    paths, rows = {}, defaultdict(dict)
    cat_names = {c["id"]: c["name"] for c in train_coco["categories"]}
    for f in sorted(fractions):
        n = max(1, round(f * len(img_ids)))
        keep = set(order[:n])
        sub = {"images": [im for im in train_coco["images"] if im["id"] in keep],
               "annotations": [a for a in train_coco["annotations"] if a["image_id"] in keep],
               "categories": train_coco["categories"]}
        p = os.path.join(out_dir, f"train_{frac_tag(f)}.json")
        save_json(sub, p)
        paths[f] = p
        cnt = Counter(a["category_id"] for a in sub["annotations"])
        for cid, name in cat_names.items():
            rows[name][f"inst_{frac_tag(f)}"] = cnt.get(cid, 0)
        n_cls = sum(1 for c in cat_names if cnt.get(c, 0) > 0)
        print(f"  subset {frac_tag(f)}: {len(sub['images'])} gambar, {len(sub['annotations'])} anotasi, "
              f"{n_cls}/{len(cat_names)} kelas punya contoh")
    with open(os.path.join(out_dir, "subset_class_counts.csv"), "w", newline="", encoding="utf-8") as fh:
        cols = [f"inst_{frac_tag(f)}" for f in sorted(fractions)]
        w = csv.writer(fh)
        w.writerow(["class"] + cols)
        for name in sorted(rows, key=lambda n: -rows[n][cols[-1]]):
            w.writerow([name] + [rows[name][c] for c in cols])
    if n_cover > round(min(fractions) * len(img_ids)):
        print(f"  PERINGATAN: butuh {n_cover} gambar untuk mencakup semua kelas, lebih dari subset terkecil.")
    return paths


def limit_coco(coco_path, n, out_path, seed=0):
    """Ambil n gambar pertama (untuk smoke test)."""
    c = load_json(coco_path)
    ids = sorted(im["id"] for im in c["images"])
    random.Random(seed).shuffle(ids)
    keep = set(ids[:n])
    c["images"] = [im for im in c["images"] if im["id"] in keep]
    c["annotations"] = [a for a in c["annotations"] if a["image_id"] in keep]
    save_json(c, out_path)
    return out_path


# ----------------------------------------------------------------------------- persiapan data lengkap
def prepare_data(cfg):
    """Download v5 (+ foto sendiri), seragamkan kelas, buat subset. Mengembalikan dict path."""
    data_dir = cfg["DATA_DIR"]
    canon = os.path.join(cfg["RESULTS_DIR"], "data_canonical")
    os.makedirs(canon, exist_ok=True)
    api_key = None

    v5 = cfg.get("V5_DIR")
    if not has_coco_split(v5):
        api_key = api_key or get_secret("ROBOFLOW_API_KEY")
        v5 = download_roboflow(api_key, "panopticindoor", "panoptic-indoor-segmentation", 5,
                               os.path.join(data_dir, "panoptic-indoor-v5-coco"))
    merge = cfg.get("CLASS_MERGE") or {}
    classes = canonical_classes(os.path.join(v5, "train", "_annotations.coco.json"), merge)
    print(f"Jumlah kelas: {len(classes)}")
    save_json(classes, os.path.join(canon, "classes.json"))

    out = {"classes": classes, "splits": {}}
    resized = os.path.join(data_dir, "resized")
    for split in ("train", "valid", "test"):
        name = "test_v5" if split == "test" else split
        p = os.path.join(canon, f"{name}.json")
        to_canonical_coco([os.path.join(v5, split)], classes, p, merge, resized, max_side=10 ** 9)  # v5 tidak diubah
        out["splits"][name] = p

    # Test set foto sendiri (opsional: dilewati kalau belum ada / gagal download)
    mc = cfg.get("MYCAPTURE") or {}
    if mc.get("ENABLED"):
        try:
            d = mc.get("DIR")
            if not has_coco_split(d):
                api_key = api_key or get_secret("ROBOFLOW_API_KEY")
                d = download_roboflow(api_key, mc["WORKSPACE"], mc["PROJECT"], mc["VERSION"],
                                      os.path.join(data_dir, "mycapture-coco"))
            split_dirs = [os.path.join(d, s) for s in ("train", "valid", "test")
                          if os.path.exists(os.path.join(d, s, "_annotations.coco.json"))]
            p = os.path.join(canon, "mycapture.json")
            c = to_canonical_coco(split_dirs, classes, p, merge, resized)
            print(f"  mycapture: {len(c['images'])} gambar, {len(c['annotations'])} anotasi")
            out["splits"]["mycapture"] = p
        except Exception as e:
            print("  PERINGATAN: test set foto sendiri dilewati ->", repr(e))

    if cfg.get("SMOKE"):
        for k in list(out["splits"]):
            n = 12 if k == "train" else 6
            out["splits"][k] = limit_coco(out["splits"][k], n, out["splits"][k].replace(".json", "_smoke.json"))

    train = load_json(out["splits"]["train"])
    for k, p in out["splits"].items():
        c = load_json(p)
        print(f"  {k:10s}: {len(c['images']):4d} gambar, {len(c['annotations']):5d} anotasi")
    out["subsets"] = make_nested_subsets(train, cfg["FRACTIONS"], cfg["SEED"], os.path.join(cfg["RESULTS_DIR"], "subsets"))
    return out


# ----------------------------------------------------------------------------- evaluasi
def mask_to_rle(binary):
    from pycocotools import mask as M
    rle = M.encode(np.asfortranarray(binary.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle


STAT_NAMES = ["AP", "AP50", "AP75", "APs", "APm", "APl"]


def coco_evaluate(gt_path, preds, max_det, iou_type="segm", img_ids=None):
    """mAP COCO via pycocotools. Return (dict ringkasan, dict AP per kelas).
    img_ids: kalau diisi, evaluasi hanya pada gambar-gambar itu (mis. per kondisi foto)."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(gt_path)
    names = {c["id"]: c["name"] for c in gt.loadCats(gt.getCatIds())}
    if not preds:
        return {k: 0.0 for k in STAT_NAMES}, {}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes([dict(p) for p in preds])
        E = COCOeval(gt, dt, iou_type)
        E.params.maxDets = [1, 10, max_det]
        if img_ids is not None:
            E.params.imgIds = sorted(img_ids)
        E.evaluate()
        E.accumulate()
    # Dihitung langsung dari tabel precision (E.stats bawaan pycocotools selalu memakai maxDets=100).
    prec = E.eval["precision"]  # [T(IoU), R(recall), K(kelas), A(area: all,s,m,l), M(maxDets)]
    ious = E.params.iouThrs

    def _ap(p):
        return float(np.mean(p[p > -1])) if (p > -1).any() else 0.0

    summary = {
        "AP": _ap(prec[:, :, :, 0, -1]),
        "AP50": _ap(prec[np.isclose(ious, 0.5)][:, :, :, 0, -1]),
        "AP75": _ap(prec[np.isclose(ious, 0.75)][:, :, :, 0, -1]),
        "APs": _ap(prec[:, :, :, 1, -1]),
        "APm": _ap(prec[:, :, :, 2, -1]),
        "APl": _ap(prec[:, :, :, 3, -1]),
    }
    per_class = {}
    for k_idx, cid in enumerate(E.params.catIds):
        p = prec[:, :, k_idx, 0, -1]
        if (p > -1).any():
            per_class[names[cid]] = float(np.mean(p[p > -1]))
    return summary, per_class


def evaluate_and_log(model_name, tag, split, gt_path, preds, run_dir, results_dir, max_det, extra=None):
    segm, per_class = coco_evaluate(gt_path, preds, max_det, "segm")
    bbox, _ = coco_evaluate(gt_path, preds, max_det, "bbox")
    row = {"model": model_name, "run": tag, "split": split, "n_pred": len(preds)}
    row.update({f"mask_{k}": round(v, 4) for k, v in segm.items()})
    row.update({f"box_{k}": round(v, 4) for k, v in bbox.items()})
    row.update(extra or {})
    append_csv(os.path.join(results_dir, "metrics.csv"), row)
    with open(os.path.join(run_dir, f"per_class_AP_{split}.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["class", "mask_AP"])
        for n, v in sorted(per_class.items(), key=lambda kv: -kv[1]):
            w.writerow([n, round(v, 4)])
    print(f"  {split:10s} mask AP={segm['AP']:.4f} AP50={segm['AP50']:.4f} | box AP={bbox['AP']:.4f}")
    return row


# ----------------------------------------------------------------------------- visualisasi
_PALETTE = np.array([[230, 25, 75], [60, 180, 75], [255, 225, 25], [0, 130, 200], [245, 130, 48],
                     [145, 30, 180], [70, 240, 240], [240, 50, 230], [210, 245, 60], [250, 190, 212],
                     [0, 128, 128], [220, 190, 255], [170, 110, 40], [128, 0, 0], [170, 255, 195]], dtype=np.uint8)


def save_overlays(gt_path, preds, out_dir, classes, score_thr=0.3, max_images=60):
    """Simpan gambar dengan mask prediksi (skor >= score_thr) untuk analisis kegagalan."""
    from pycocotools import mask as M
    from PIL import ImageDraw
    os.makedirs(out_dir, exist_ok=True)
    gt = load_json(gt_path)
    by_img = defaultdict(list)
    for p in preds:
        if p["score"] >= score_thr:
            by_img[p["image_id"]].append(p)
    for im in gt["images"][:max_images]:
        img = np.array(Image.open(im["path"]).convert("RGB")).astype(np.float32)
        labels = []
        for j, p in enumerate(sorted(by_img[im["id"]], key=lambda p: -p["score"])):
            m = M.decode(p["segmentation"]).astype(bool)
            if m.shape != img.shape[:2]:
                continue
            col = _PALETTE[(p["category_id"] - 1) % len(_PALETTE)]
            img[m] = img[m] * 0.5 + col * 0.5
            ys, xs = np.nonzero(m)
            if len(xs):
                labels.append((int(xs.min()), int(ys.min()), f"{classes[p['category_id'] - 1]} {p['score']:.2f}", col))
        pil = Image.fromarray(img.clip(0, 255).astype(np.uint8))
        dr = ImageDraw.Draw(pil)
        for x, y, t, col in labels:
            dr.rectangle([x, y, x + 7 * len(t), y + 12], fill=tuple(int(c) for c in col))
            dr.text((x + 1, y), t, fill=(0, 0, 0))
        pil.save(os.path.join(out_dir, os.path.splitext(im.get("orig_file_name", im["file_name"]))[0] + "_pred.jpg"), quality=90)


def run_predictions_and_eval(model_name, tag, predict_fn, data, cfg, run_dir, extra=None):
    """Prediksi + evaluasi di valid, test_v5, dan mycapture (kalau ada). predict_fn(coco_dict) -> list pred."""
    rows = []
    for split in ("valid", "test_v5", "mycapture"):
        if split not in data["splits"]:
            continue
        gt_path = data["splits"][split]
        t0 = time.time()
        preds = predict_fn(load_json(gt_path))
        if split != "valid":
            save_json(preds, os.path.join(run_dir, f"preds_{split}.json"))
        ex = dict(extra or {})
        ex["infer_sec"] = round(time.time() - t0, 1)
        rows.append(evaluate_and_log(model_name, tag, split, gt_path, preds, run_dir, cfg["RESULTS_DIR"],
                                     cfg["MAX_DET"], ex))
        if split == "mycapture":
            save_overlays(gt_path, preds, os.path.join(run_dir, "overlays_mycapture"), data["classes"])
    return rows


# ----------------------------------------------------------------------------- jadwal LR untuk model PyTorch
class TimeAwareCosine:
    """Cosine LR dengan warmup. Progress = max(iterasi/total, waktu/batas_waktu), jadi kalau batas
    waktu tercapai lebih dulu, jadwal LR tetap selesai turun (adil antar run)."""

    def __init__(self, optimizer, total_iters, warmup_iters, time_limit_sec):
        self.opt, self.total, self.warm, self.limit = optimizer, max(1, total_iters), warmup_iters, time_limit_sec
        self.base = [g["lr"] for g in optimizer.param_groups]
        self.t0, self.it = time.time(), 0

    def progress(self):
        p = self.it / self.total
        if self.limit:
            p = max(p, (time.time() - self.t0) / self.limit)
        return min(p, 1.0)

    def step(self):
        self.it += 1
        f = 0.5 * (1 + math.cos(math.pi * self.progress()))
        if self.it < self.warm:
            f *= self.it / self.warm
        for g, b in zip(self.opt.param_groups, self.base):
            g["lr"] = b * f

    def done(self):
        return self.progress() >= 1.0


def zip_results(results_dir):
    base = results_dir.rstrip("/\\")
    shutil.make_archive(base, "zip", results_dir)
    print("Hasil dikompres:", base + ".zip")


In [ ]:
import os, sys, gc, time, json
sys.path.insert(0, os.getcwd())
from common import *

CFG = dict(
    SEED=42,
    FRACTIONS=[0.25, 0.5, 1.0],          # variabel NRP digit 2-3
    TIME_LIMIT_MIN_PER_RUN=40,           # 3 run x 40 menit = 2 jam GPU per model (sesuai catatan tugas)
    MAX_DET=200,                         # ada gambar test dengan 129 objek, jadi batas 100 terlalu kecil
    SCORE_THRESH=0.05,                   # prediksi dengan skor di bawah ini dibuang (sama untuk semua model)
    CLASS_MERGE={},                      # opsional, contoh: {"scissor": "scissors", "wall": "wall-other-merged", "st-": None}
    V5_DIR=None,                         # isi path kalau dataset v5 sudah ada (mis. di-upload sebagai Kaggle Dataset)
    MYCAPTURE=dict(
        ENABLED=True,
        WORKSPACE="fpkkgunakanhelmdataset",   # dari URL Roboflow: app.roboflow.com/<workspace>/<project>
        PROJECT="room_mycapture",
        VERSION=1,                            # nomor versi hasil "Generate Version" di Roboflow
        DIR=None,                             # atau path folder export COCO kalau sudah di-upload ke Kaggle
    ),
    RUN_TRAIN=True,                      # False = tidak training ulang, pakai bobot dari WEIGHTS_FROM
    WEIGHTS_FROM=None,                   # mis. "/kaggle/input/<output-notebook-sebelumnya>/results_mask2former"
    SMOKE=bool(int(os.environ.get("SMOKE", "0"))),   # uji cepat (beberapa gambar, 1 epoch)
)
CFG["WORK_DIR"] = "/kaggle/working" if ON_KAGGLE else os.path.abspath("work")
CFG["DATA_DIR"] = "/tmp/data" if ON_KAGGLE else os.path.join(CFG["WORK_DIR"], "data")
CFG["RESULTS_DIR"] = os.path.join(CFG["WORK_DIR"], "results_mask2former")
WORKERS = 0 if ON_WINDOWS else 4
os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
save_json(CFG, os.path.join(CFG["RESULTS_DIR"], "config.json"))
seed_everything(CFG["SEED"])

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", gpu_name())

In [ ]:
DATA = prepare_data(CFG)
CLASSES = DATA["classes"]
NUM_CLASSES = len(CLASSES)

## Dataset, model, training

In [ ]:
from pycocotools import mask as M


def decode_masks(anns, h, w, scale=1.0):
    """Polygon/RLE COCO -> (N, h', w') uint8 dan label 1..K. scale mengecilkan/membesarkan koordinat."""
    nh, nw = round(h * scale), round(w * scale)
    masks, labels = [], []
    for a in anns:
        seg = a["segmentation"]
        if isinstance(seg, list):
            rle = M.merge(M.frPyObjects([[v * scale for v in p] for p in seg], nh, nw))
            m = M.decode(rle)
        else:
            rle = M.frPyObjects(seg, h, w) if isinstance(seg["counts"], list) else seg
            m = M.decode(rle)
            if scale != 1.0:
                m = np.array(Image.fromarray(m * 255).resize((nw, nh), Image.NEAREST)) > 127
        if m.any():
            masks.append(m.astype(np.uint8))
            labels.append(a["category_id"])
    if masks:
        return np.stack(masks), np.array(labels, dtype=np.int64)
    return np.zeros((0, nh, nw), np.uint8), np.zeros((0,), np.int64)


import random as _random
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor

MODEL_NAME = "mask2former-swin-tiny"
CKPT = "facebook/mask2former-swin-tiny-coco-instance"   # pretrained COCO instance
LONG_SIDE = 640                          # sama dengan model lain
EPOCHS = 1 if CFG["SMOKE"] else 60
BATCH = 1 if CFG["SMOKE"] else 4
LR, BACKBONE_LR_MULT, WD = 1e-4, 0.1, 0.05
WARMUP_ITERS = 100
EVAL_EVERY = 5
PATIENCE_EVALS = 4
USE_AMP = DEVICE == "cuda"

_proc = AutoImageProcessor.from_pretrained(CKPT)
MEAN, STD = np.array(_proc.image_mean, dtype=np.float32), np.array(_proc.image_std, dtype=np.float32)
if MEAN.max() > 1:
    MEAN, STD = MEAN / 255, STD / 255


def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    s = LONG_SIDE / max(w, h)
    nw, nh = round(w * s), round(h * s)
    arr = (np.asarray(img.resize((nw, nh), Image.BILINEAR), dtype=np.float32) / 255 - MEAN) / STD
    return torch.from_numpy(arr).permute(2, 0, 1), s, (h, w)


class M2FDataset(Dataset):
    def __init__(self, coco_path, train):
        c = load_json(coco_path)
        self.by = defaultdict(list)
        for a in c["annotations"]:
            self.by[a["image_id"]].append(a)
        self.images = [im for im in c["images"] if self.by[im["id"]]] if train else c["images"]
        self.train = train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        im = self.images[i]
        x, s, (h, w) = load_image(im["path"])
        masks, labels = decode_masks(self.by[im["id"]], h, w, s)
        masks = torch.from_numpy(masks[:, : x.shape[1], : x.shape[2]])
        if self.train and _random.random() < 0.5:
            x, masks = x.flip(-1), masks.flip(-1)
        return x, masks, torch.from_numpy(labels - 1)


def pad_batch(xs):
    H = max(x.shape[1] for x in xs)
    W = max(x.shape[2] for x in xs)
    H, W = (H + 31) // 32 * 32, (W + 31) // 32 * 32
    pv = torch.zeros(len(xs), 3, H, W)
    pm = torch.zeros(len(xs), H, W, dtype=torch.long)
    for i, x in enumerate(xs):
        pv[i, :, : x.shape[1], : x.shape[2]] = x
        pm[i, : x.shape[1], : x.shape[2]] = 1
    return pv, pm


def collate(batch):
    xs, ms, ls = zip(*batch)
    pv, pm = pad_batch(xs)
    H, W = pv.shape[-2:]
    mask_labels = []
    for m in ms:
        out = torch.zeros(m.shape[0], H, W)
        out[:, : m.shape[1], : m.shape[2]] = m.float()
        mask_labels.append(out)
    return pv, pm, mask_labels, list(ls)


def build_model():
    return Mask2FormerForUniversalSegmentation.from_pretrained(
        CKPT, id2label={i: n for i, n in enumerate(CLASSES)}, label2id={n: i for i, n in enumerate(CLASSES)},
        ignore_mismatched_sizes=True).to(DEVICE)


@torch.no_grad()
def predict(model, coco):
    model.eval()
    preds = []
    for im in coco["images"]:
        x, s, (h, w) = load_image(im["path"])
        pv, pm = pad_batch([x])
        with torch.autocast("cuda", enabled=USE_AMP):
            out = model(pixel_values=pv.to(DEVICE), pixel_mask=pm.to(DEVICE))
        cls = out.class_queries_logits[0].float().softmax(-1)[:, :-1]       # (Q, K)
        Q, K = cls.shape
        scores, idx = cls.flatten().topk(min(CFG["MAX_DET"], Q * K))
        q, lab = idx // K, idx % K
        ml = out.masks_queries_logits[0].float()[q]                          # (D, H/4, W/4)
        ml = F.interpolate(ml[None], size=pv.shape[-2:], mode="bilinear", align_corners=False)[0]
        ml = ml[:, : x.shape[1], : x.shape[2]]
        # jumlah mask per potongan disesuaikan ukuran gambar: test v5 punya gambar ~10 MP,
        # 50 mask sekaligus di resolusi asli butuh > 3 GB dan bikin CUDA OOM
        chunk = max(1, min(50, int(4e7 // (h * w))))
        for j in range(0, len(ml), chunk):
            prob = F.interpolate(ml[None, j: j + chunk], size=(h, w), mode="bilinear", align_corners=False)[0].sigmoid()
            binary = prob > 0.5
            area = binary.flatten(1).sum(1, dtype=torch.float32)
            mscore = (prob * binary).flatten(1).sum(1) / (area + 1e-6)
            final = scores[j: j + chunk] * mscore
            for k in range(len(prob)):
                if area[k] > 0 and final[k] >= CFG["SCORE_THRESH"]:
                    preds.append({"image_id": im["id"], "category_id": int(lab[j + k]) + 1,
                                  "segmentation": mask_to_rle(binary[k].cpu().numpy()), "score": float(final[k])})
    return preds


def load_trained(run_dir):
    m = build_model()
    m.load_state_dict(torch.load(os.path.join(run_dir, "best.pt"), map_location=DEVICE, weights_only=True))
    return m


def train_one(train_json, valid_json, run_dir):
    dl = DataLoader(M2FDataset(train_json, train=True), batch_size=BATCH, shuffle=True,
                    num_workers=WORKERS, collate_fn=collate)
    model = build_model()
    bb = [p for n, p in model.named_parameters() if "pixel_level_module.encoder" in n and p.requires_grad]
    rest = [p for n, p in model.named_parameters() if "pixel_level_module.encoder" not in n and p.requires_grad]
    opt = torch.optim.AdamW([{"params": bb, "lr": LR * BACKBONE_LR_MULT}, {"params": rest, "lr": LR}], weight_decay=WD)
    limit = None if CFG["SMOKE"] else CFG["TIME_LIMIT_MIN_PER_RUN"] * 60
    sched = TimeAwareCosine(opt, EPOCHS * len(dl), WARMUP_ITERS, limit)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    valid = load_json(valid_json)
    best_ap, bad, best_ep, ep = -1.0, 0, 0, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        tot, n, skipped, t0 = 0.0, 0, 0, time.time()
        for pv, pm, ml, cl in dl:
            with torch.autocast("cuda", enabled=USE_AMP):
                out = model(pixel_values=pv.to(DEVICE), pixel_mask=pm.to(DEVICE),
                            mask_labels=[m.to(DEVICE) for m in ml], class_labels=[c.to(DEVICE) for c in cl])
            loss = out.loss
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)   # sesuai konfigurasi asli Mask2Former
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()
            n += 1
            if sched.done():
                break
        row = {"epoch": ep, "lr": opt.param_groups[1]["lr"], "loss": round(tot / max(n, 1), 4),
               "skipped": skipped, "sec": round(time.time() - t0, 1)}
        last = ep == EPOCHS or sched.done()
        if ep % EVAL_EVERY == 0 or last:
            segm, _ = coco_evaluate(valid_json, predict(model, valid), CFG["MAX_DET"])
            row.update({"val_mask_AP": round(segm["AP"], 4), "val_mask_AP50": round(segm["AP50"], 4)})
            if segm["AP"] > best_ap:
                best_ap, bad, best_ep = segm["AP"], 0, ep
                torch.save(model.state_dict(), os.path.join(run_dir, "best.pt"))
            else:
                bad += 1
        append_csv(os.path.join(run_dir, "results.csv"), row)
        print(row)
        if last or bad >= PATIENCE_EVALS:
            break
    return load_trained(run_dir), {"epochs_run": ep, "best_epoch": best_ep}

## Training 25% → 50% → 100% + evaluasi

In [ ]:
for frac in CFG["FRACTIONS"]:
    tag = frac_tag(frac)
    run_dir = os.path.join(CFG["RESULTS_DIR"], tag)
    os.makedirs(run_dir, exist_ok=True)
    seed_everything(CFG["SEED"])
    print(f"\n===== {MODEL_NAME} | train {tag} =====")
    if CFG["RUN_TRAIN"]:
        with GpuTimer(CFG["RESULTS_DIR"], MODEL_NAME, tag, note="train") as tm:
            model, info = train_one(DATA["subsets"][frac], DATA["splits"]["valid"], run_dir)
        info["train_min"] = round(tm.minutes, 1)
    else:
        model, info = load_trained(os.path.join(CFG["WEIGHTS_FROM"], tag)), {}
    with GpuTimer(CFG["RESULTS_DIR"], MODEL_NAME, tag, note="eval"):
        run_predictions_and_eval(MODEL_NAME, tag, lambda c: predict(model, c), DATA, CFG, run_dir, info)
    del model
    gc.collect()
    torch.cuda.empty_cache()

## Ringkasan

In [ ]:
import pandas as pd
m = pd.read_csv(os.path.join(CFG["RESULTS_DIR"], "metrics.csv"))
print(m.pivot_table(index="run", columns="split", values=["mask_AP", "mask_AP50"]).round(4).to_string())
print()
print(pd.read_csv(os.path.join(CFG["RESULTS_DIR"], "gpu_log.csv")).to_string())
zip_results(CFG["RESULTS_DIR"])

### Isi folder hasil
- `metrics.csv` : mask/box AP setiap run (25/50/100%) di valid, test_v5, mycapture
- `gpu_log.csv` : catatan waktu GPU (bukti proses)
- `subsets/` : daftar gambar tiap subset + `subset_class_counts.csv` (kelas yang hilang di subset kecil)
- `<run>/results.csv` : log training per epoch
- `<run>/per_class_AP_<split>.csv` : AP per kelas (bahan analisis kelas tersulit)
- `<run>/preds_<split>.json` : prediksi format COCO (untuk analisis kegagalan)
- `<run>/overlays_mycapture/` : gambar fotomu dengan mask prediksi